# Retrieval-Augmented Generation (RAG)

## What We'll Build

In this notebook, we'll build a complete **Retrieval-Augmented Generation (RAG)** system from scratch. RAG is one of the most important techniques for production LLM applications.

**What we'll learn:**
- Why RAG solves critical LLM limitations (hallucinations, knowledge cutoff)
- How to chunk documents effectively
- How to generate and store embeddings
- How to build a vector search engine from scratch
- How to combine retrieval with generation
- Advanced retrieval strategies (hybrid search, re-ranking)
- Production considerations and optimizations

**Why this matters:**

RAG enables LLMs to:
- Access up-to-date information beyond their training cutoff
- Answer questions about private/proprietary documents
- Provide citations and source attribution
- Reduce hallucinations by grounding responses in retrieved facts

This is a **TIER 11** notebook - essential knowledge for building production LLM systems.

## 1. Introduction: The LLM Knowledge Problem

### Why Do We Need RAG?

Large Language Models (LLMs) have three critical limitations:

1. **Knowledge Cutoff**: Training data has a fixed date - models don't know about events after training
2. **Hallucinations**: Models confidently generate plausible-sounding but incorrect information
3. **Domain Specificity**: Models lack knowledge about private documents, company data, or specialized domains

**RAG Solution**: Instead of relying solely on the model's parametric memory, we:
1. **Retrieve** relevant documents from an external knowledge base
2. **Augment** the prompt with retrieved context
3. **Generate** answers grounded in the retrieved facts

This shifts from "what does the model remember?" to "what can the model find and use?"

## 2. Setup

Let's import the libraries we'll need and configure our environment.

In [ ]:


%load_ext autoreload
%autoreload 2

### Set Random Seed for Reproducibility

We'll set a random seed to ensure consistent results across runs.

In [ ]:
from aiml_notebooks import get_device, set_seed

In [ ]:
set_seed(42)
device = get_device(prefer_cpu=True)  # Prefer CPU for transformer compatibility
print(f"Using device: {device}")

### Install Sentence Transformers

We'll use **sentence-transformers** for generating high-quality text embeddings. This library provides pre-trained models optimized for semantic similarity tasks.

In [ ]:
# Import sentence transformers (installed via uv)
from sentence_transformers import SentenceTransformer

print("Sentence transformers imported successfully!")

## 3. RAG Pipeline Overview

### The Six Stages of RAG

A complete RAG system has six stages:

1. **Document Ingestion**: Load and preprocess documents
2. **Chunking**: Split documents into smaller, semantically meaningful pieces
3. **Embedding Generation**: Convert text chunks to dense vectors
4. **Vector Storage & Indexing**: Store embeddings for efficient retrieval
5. **Retrieval**: Find most relevant chunks for a query
6. **Augmented Generation**: Use retrieved context to generate answers

Let's visualize this pipeline:

In [ ]:
# Visualize the RAG pipeline
fig, ax = plt.subplots(figsize=(14, 6))
ax.axis('off')

stages = [
    "1. Documents",
    "2. Chunks",
    "3. Embeddings",
    "4. Vector DB",
    "5. Retrieval",
    "6. Generation"
]

# Draw stages
for i, stage in enumerate(stages):
    x = i * 2.2
    rect = plt.Rectangle((x, 0), 2, 1, facecolor='lightblue', edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x + 1, 0.5, stage, ha='center', va='center', fontsize=10, fontweight='bold')
    
    # Draw arrows
    if i < len(stages) - 1:
        ax.annotate('', xy=(x + 2.2, 0.5), xytext=(x + 2, 0.5),
                   arrowprops=dict(arrowstyle='->', lw=2, color='black'))

ax.set_xlim(-0.5, 13)
ax.set_ylim(-0.5, 1.5)
ax.set_title('RAG Pipeline: From Documents to Answers', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("This is the end-to-end RAG pipeline we'll build!")

## 4. Document Collection

### Creating a Sample Knowledge Base

Let's create a small collection of documents about AI/ML topics. In production, you'd load documents from files, databases, or APIs.

In [ ]:
# Sample documents about AI/ML topics
documents = [
    {
        "id": "doc1",
        "title": "Transformers Architecture",
        "content": """The Transformer architecture was introduced in the paper 'Attention Is All You Need' by Vaswani et al. in 2017. 
        It revolutionized NLP by replacing recurrent layers with self-attention mechanisms. The key innovation is multi-head attention, 
        which allows the model to focus on different parts of the input sequence simultaneously. Transformers consist of encoder and 
        decoder stacks, each containing multiple layers of attention and feed-forward networks. The architecture uses positional 
        encodings to maintain sequence order information. Modern LLMs like GPT, BERT, and T5 are all based on the Transformer architecture."""
    },
    {
        "id": "doc2",
        "title": "Attention Mechanism",
        "content": """Attention mechanisms allow neural networks to focus on relevant parts of the input when producing output. 
        The attention function can be described as mapping a query and key-value pairs to an output. In self-attention, 
        queries, keys, and values all come from the same sequence. The attention weight is computed using the dot product 
        of query and key vectors, scaled by the square root of the dimension, then normalized with softmax. Multi-head attention 
        runs multiple attention operations in parallel, allowing the model to attend to different representation subspaces."""
    },
    {
        "id": "doc3",
        "title": "Embeddings",
        "content": """Word embeddings are dense vector representations of words that capture semantic meaning. Words with similar 
        meanings have similar embeddings in vector space. Popular embedding methods include Word2Vec, GloVe, and FastText. 
        Modern approaches use contextual embeddings from models like BERT, where the same word has different embeddings 
        depending on context. Embeddings typically have dimensions ranging from 50 to 1024. The quality of embeddings 
        significantly impacts downstream task performance. Sentence embeddings extend this concept to entire sentences or paragraphs."""
    },
    {
        "id": "doc4",
        "title": "RAG Systems",
        "content": """Retrieval-Augmented Generation combines information retrieval with text generation. RAG systems first retrieve 
        relevant documents from a knowledge base, then use those documents to augment the input to a language model. This approach 
        helps reduce hallucinations and allows models to access information beyond their training data. RAG is particularly useful 
        for question answering, fact-checking, and domain-specific applications. The retrieval component typically uses dense 
        embeddings and vector similarity search. Advanced RAG systems may use hybrid retrieval combining dense and sparse methods."""
    },
    {
        "id": "doc5",
        "title": "Vector Databases",
        "content": """Vector databases are specialized systems for storing and querying high-dimensional vectors. They use approximate 
        nearest neighbor (ANN) algorithms like HNSW or IVF for efficient similarity search. Popular vector databases include 
        Pinecone, Weaviate, Milvus, and Chroma. These systems can handle millions or billions of vectors while maintaining 
        low-latency retrieval. Vector databases often support metadata filtering and hybrid search combining vector similarity 
        with traditional filters. They are essential infrastructure for RAG systems at scale."""
    },
    {
        "id": "doc6",
        "title": "Fine-tuning LLMs",
        "content": """Fine-tuning adapts pre-trained language models to specific tasks or domains. Full fine-tuning updates all model 
        parameters, while parameter-efficient methods like LoRA only update a small subset. Fine-tuning requires labeled data 
        and careful hyperparameter selection to avoid catastrophic forgetting. The learning rate should be much smaller than 
        pre-training to preserve general knowledge. Common approaches include supervised fine-tuning on task-specific data, 
        instruction tuning for following commands, and reinforcement learning from human feedback (RLHF) for alignment."""
    }
]

print(f"Created knowledge base with {len(documents)} documents")
print("\nSample document:")
print(f"Title: {documents[0]['title']}")
print(f"Content length: {len(documents[0]['content'])} characters")

## 5. Document Chunking

### Why Chunk Documents?

**Chunking** splits long documents into smaller pieces. This is critical because:

1. **Embedding quality**: Shorter texts produce more focused, meaningful embeddings
2. **Retrieval precision**: Small chunks allow fine-grained matching to queries
3. **Context window limits**: LLMs have limited input lengths (e.g., 4096 tokens)
4. **Relevance**: Entire documents often contain mixed topics; chunks are more targeted

**Trade-offs**:
- **Small chunks**: More precise, but may lack context
- **Large chunks**: More context, but less precise retrieval
- **Overlap**: Helps preserve context across boundaries, but increases storage

### Sentence-Based Chunking

Let's implement a simple but effective chunking strategy: split on sentences, then group into chunks of target size.

In [ ]:
from typing import List, Dict, Tuple, Optionalfrom dataclasses import dataclass

In [ ]:
@dataclass
class Chunk:
    """Represents a text chunk with metadata."""
    text: str
    doc_id: str
    doc_title: str
    chunk_index: int
    
def split_into_sentences(text: str) -> List[str]:
    """Split text into sentences using simple regex."""
    # Simple sentence splitter - in production, use spaCy or NLTK
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sentences if s.strip()]

def chunk_text(text: str, doc_id: str, doc_title: str, 
               max_chunk_size: int = 200, 
               overlap_sentences: int = 1) -> List[Chunk]:
    """Chunk text by grouping sentences up to max_chunk_size characters."""
    sentences = split_into_sentences(text)
    chunks = []
    current_chunk = []
    current_size = 0
    
    for sentence in sentences:
        sentence_size = len(sentence)
        
        # If adding this sentence would exceed max size, save current chunk
        if current_size + sentence_size > max_chunk_size and current_chunk:
            chunk_text = ' '.join(current_chunk)
            chunks.append(Chunk(
                text=chunk_text,
                doc_id=doc_id,
                doc_title=doc_title,
                chunk_index=len(chunks)
            ))
            
            # Keep last N sentences for overlap
            current_chunk = current_chunk[-overlap_sentences:] if overlap_sentences > 0 else []
            current_size = sum(len(s) for s in current_chunk)
        
        current_chunk.append(sentence)
        current_size += sentence_size
    
    # Add final chunk
    if current_chunk:
        chunk_text = ' '.join(current_chunk)
        chunks.append(Chunk(
            text=chunk_text,
            doc_id=doc_id,
            doc_title=doc_title,
            chunk_index=len(chunks)
        ))
    
    return chunks

print("Chunking functions defined!")

### Apply Chunking to All Documents

Now let's chunk our entire document collection and examine the results.

In [ ]:
# Chunk all documents
all_chunks = []
for doc in documents:
    doc_chunks = chunk_text(
        text=doc['content'],
        doc_id=doc['id'],
        doc_title=doc['title'],
        max_chunk_size=200,
        overlap_sentences=1
    )
    all_chunks.extend(doc_chunks)

print(f"Total chunks created: {len(all_chunks)}")
print(f"Average chunk length: {np.mean([len(c.text) for c in all_chunks]):.1f} characters")

# Show some examples
print("\nExample chunks:")
for i in range(3):
    chunk = all_chunks[i]
    print(f"\nChunk {i+1} (from {chunk.doc_title}):")
    print(f"  Text: {chunk.text[:150]}...")
    print(f"  Length: {len(chunk.text)} chars")

### Visualize Chunk Size Distribution

Let's examine how our chunking strategy distributed text across chunks.

In [ ]:
# Plot chunk size distribution
chunk_sizes = [len(c.text) for c in all_chunks]

plt.figure(figsize=(10, 5))
plt.hist(chunk_sizes, bins=20, edgecolor='black', alpha=0.7)
plt.axvline(200, color='red', linestyle='--', linewidth=2, label='Max chunk size')
plt.axvline(np.mean(chunk_sizes), color='green', linestyle='--', linewidth=2, label=f'Mean ({np.mean(chunk_sizes):.0f})')
plt.xlabel('Chunk Size (characters)')
plt.ylabel('Frequency')
plt.title('Distribution of Chunk Sizes')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Min: {min(chunk_sizes)}, Max: {max(chunk_sizes)}, Mean: {np.mean(chunk_sizes):.1f}")

## 6. Embeddings: Converting Text to Vectors

### What Are Embeddings?

**Embeddings** are dense vector representations of text that capture semantic meaning. Key properties:

- **Semantic similarity**: Similar texts have similar embeddings (high cosine similarity)
- **Dense vectors**: Typically 384-1024 dimensions
- **Learned representations**: Trained on large corpora to capture meaning

**Why sentence-transformers?**
- Pre-trained on semantic similarity tasks
- Optimized for retrieval (better than GPT embeddings for search)
- Fast inference
- Various model sizes (all-MiniLM-L6-v2 is small and fast)

### Load Embedding Model

We'll use **all-MiniLM-L6-v2**, a popular model that balances quality and speed. It produces 384-dimensional embeddings.

In [ ]:
# Load embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Test with a simple example
test_text = "This is a test sentence."
test_embedding = embedding_model.encode(test_text)

print(f"Embedding model loaded: all-MiniLM-L6-v2")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"Embedding shape: {test_embedding.shape}")
print(f"Sample values: {test_embedding[:5]}")

### Generate Embeddings for All Chunks

Now let's embed all our text chunks. This creates our **vector representation** of the knowledge base.

In [ ]:
# Extract chunk texts
chunk_texts = [chunk.text for chunk in all_chunks]

# Generate embeddings for all chunks (batch encoding is efficient)
print(f"Encoding {len(chunk_texts)} chunks...")
chunk_embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True, convert_to_numpy=True)

print(f"\nEmbeddings generated!")
print(f"Embedding matrix shape: {chunk_embeddings.shape}")
print(f"Memory usage: {chunk_embeddings.nbytes / 1024:.1f} KB")

### Visualize Embedding Similarity

Let's compute pairwise similarities between the first few chunks to see how embeddings capture semantic relationships.

In [ ]:
# Compute cosine similarity between first 8 chunks
n_display = min(8, len(chunk_embeddings))
sample_embeddings = chunk_embeddings[:n_display]

# Normalize embeddings for cosine similarity
normalized = sample_embeddings / np.linalg.norm(sample_embeddings, axis=1, keepdims=True)
similarity_matrix = normalized @ normalized.T

# Plot heatmap
plt.figure(figsize=(10, 8))
plt.imshow(similarity_matrix, cmap='RdYlGn', vmin=0, vmax=1)
plt.colorbar(label='Cosine Similarity')
plt.title('Semantic Similarity Between Chunks', fontsize=14, fontweight='bold')
plt.xlabel('Chunk Index')
plt.ylabel('Chunk Index')

# Add similarity values
for i in range(n_display):
    for j in range(n_display):
        plt.text(j, i, f'{similarity_matrix[i, j]:.2f}', 
                ha='center', va='center', color='black', fontsize=8)

plt.tight_layout()
plt.show()

print("Notice: Chunks from the same document (e.g., 0-1, 2-3) have higher similarity!")

## 7. Vector Search: Building a Retrieval Engine

### From Embeddings to Search

Now we have embeddings, we need to **search** them efficiently. The core idea:

1. Embed the query
2. Compute similarity between query embedding and all chunk embeddings
3. Return top-k most similar chunks

**Similarity metric**: We'll use **cosine similarity**, which measures the angle between vectors (0 = orthogonal, 1 = identical direction).

### Implement Cosine Similarity Search

Let's build a simple but effective vector search function from scratch using NumPy.

In [ ]:
def cosine_similarity(query_embedding: np.ndarray, 
                     corpus_embeddings: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between query and corpus embeddings.
    
    Args:
        query_embedding: Shape (embedding_dim,)
        corpus_embeddings: Shape (num_docs, embedding_dim)
        
    Returns:
        Similarities: Shape (num_docs,)
    """
    # Normalize query
    query_norm = query_embedding / np.linalg.norm(query_embedding)
    
    # Normalize corpus
    corpus_norms = corpus_embeddings / np.linalg.norm(corpus_embeddings, axis=1, keepdims=True)
    
    # Compute dot product (cosine similarity for normalized vectors)
    similarities = corpus_norms @ query_norm
    
    return similarities

def search(query: str, 
          embedding_model: SentenceTransformer,
          corpus_embeddings: np.ndarray,
          chunks: List[Chunk],
          top_k: int = 3) -> List[Tuple[Chunk, float]]:
    """Search for most relevant chunks given a query.
    
    Returns:
        List of (chunk, similarity_score) tuples
    """
    # Embed query
    query_embedding = embedding_model.encode(query, convert_to_numpy=True)
    
    # Compute similarities
    similarities = cosine_similarity(query_embedding, corpus_embeddings)
    
    # Get top-k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    # Return chunks with scores
    results = [(chunks[idx], float(similarities[idx])) for idx in top_indices]
    
    return results

print("Vector search functions defined!")

### Test Vector Search

Let's test our search engine with a few queries and examine the results.

In [ ]:
# Test query
test_query = "What is attention mechanism in neural networks?"

results = search(
    query=test_query,
    embedding_model=embedding_model,
    corpus_embeddings=chunk_embeddings,
    chunks=all_chunks,
    top_k=3
)

print(f"Query: '{test_query}'\n")
print("Top 3 results:\n")
for i, (chunk, score) in enumerate(results, 1):
    print(f"{i}. [Score: {score:.3f}] {chunk.doc_title}")
    print(f"   {chunk.text[:200]}...\n")

### Compare Multiple Queries

Let's test with different query types to see how retrieval quality varies.

In [ ]:
# Test multiple queries
test_queries = [
    "How do transformers work?",
    "What are vector databases used for?",
    "Explain word embeddings",
    "How does RAG reduce hallucinations?"
]

for query in test_queries:
    results = search(query, embedding_model, chunk_embeddings, all_chunks, top_k=1)
    top_chunk, score = results[0]
    print(f"Query: '{query}'")
    print(f"Best match [{score:.3f}]: {top_chunk.doc_title}")
    print(f"  {top_chunk.text[:150]}...\n")

## 8. Sparse Retrieval: BM25

### Dense vs Sparse Retrieval

So far we've used **dense retrieval** (embeddings). But **sparse retrieval** methods like **BM25** are still valuable:

**Dense (embeddings)**:
- ✅ Captures semantic meaning
- ✅ Handles synonyms and paraphrasing
- ❌ Can miss exact keyword matches

**Sparse (BM25)**:
- ✅ Excellent for exact matches and rare terms
- ✅ Interpretable (term-based scoring)
- ❌ No semantic understanding

**Best practice**: Use both in a **hybrid retrieval** system!

### Implement BM25 from Scratch

BM25 (Best Matching 25) is a ranking function based on term frequency and inverse document frequency.

In [ ]:
from collections import Counter

In [ ]:
class BM25:
    """BM25 sparse retrieval implementation."""
    
    def __init__(self, corpus: List[str], k1: float = 1.5, b: float = 0.75):
        """
        Args:
            corpus: List of documents
            k1: Term frequency saturation parameter
            b: Length normalization parameter
        """
        self.k1 = k1
        self.b = b
        self.corpus = corpus
        self.corpus_size = len(corpus)
        
        # Tokenize documents
        self.doc_tokens = [self._tokenize(doc) for doc in corpus]
        
        # Compute document lengths
        self.doc_lengths = [len(tokens) for tokens in self.doc_tokens]
        self.avg_doc_length = np.mean(self.doc_lengths)
        
        # Compute IDF scores
        self.idf = self._compute_idf()
    
    def _tokenize(self, text: str) -> List[str]:
        """Simple tokenization: lowercase and split on non-alphanumeric."""
        return re.findall(r'\w+', text.lower())
    
    def _compute_idf(self) -> Dict[str, float]:
        """Compute IDF (inverse document frequency) for all terms."""
        doc_frequencies = Counter()
        
        for tokens in self.doc_tokens:
            # Count unique terms per document
            unique_tokens = set(tokens)
            doc_frequencies.update(unique_tokens)
        
        idf = {}
        for term, df in doc_frequencies.items():
            # IDF formula: log((N - df + 0.5) / (df + 0.5) + 1)
            idf[term] = np.log((self.corpus_size - df + 0.5) / (df + 0.5) + 1)
        
        return idf
    
    def score(self, query: str, doc_idx: int) -> float:
        """Compute BM25 score for a query-document pair."""
        query_tokens = self._tokenize(query)
        doc_tokens = self.doc_tokens[doc_idx]
        doc_length = self.doc_lengths[doc_idx]
        
        score = 0.0
        term_frequencies = Counter(doc_tokens)
        
        for term in query_tokens:
            if term not in self.idf:
                continue
            
            tf = term_frequencies[term]
            idf = self.idf[term]
            
            # BM25 formula
            numerator = tf * (self.k1 + 1)
            denominator = tf + self.k1 * (1 - self.b + self.b * doc_length / self.avg_doc_length)
            
            score += idf * numerator / denominator
        
        return score
    
    def search(self, query: str, top_k: int = 3) -> List[Tuple[int, float]]:
        """Search and return top-k document indices with scores."""
        scores = [self.score(query, i) for i in range(self.corpus_size)]
        top_indices = np.argsort(scores)[::-1][:top_k]
        return [(idx, scores[idx]) for idx in top_indices]

print("BM25 implementation complete!")

### Test BM25 Retrieval

Let's build a BM25 index and compare results with dense retrieval.

In [ ]:
# Build BM25 index
bm25 = BM25(chunk_texts)

# Test query with exact terms
query = "transformer architecture multi-head attention"

# BM25 results
bm25_results = bm25.search(query, top_k=3)

# Dense results
dense_results = search(query, embedding_model, chunk_embeddings, all_chunks, top_k=3)

print(f"Query: '{query}'\n")
print("BM25 (Sparse) Results:")
for i, (idx, score) in enumerate(bm25_results, 1):
    print(f"{i}. [Score: {score:.2f}] {all_chunks[idx].doc_title}")
    print(f"   {all_chunks[idx].text[:100]}...\n")

print("\nDense (Embedding) Results:")
for i, (chunk, score) in enumerate(dense_results, 1):
    print(f"{i}. [Score: {score:.3f}] {chunk.doc_title}")
    print(f"   {chunk.text[:100]}...\n")

## 9. Hybrid Retrieval

### Combining Dense and Sparse

**Hybrid retrieval** combines both approaches for better results:

1. Retrieve top-k candidates from both BM25 and dense search
2. Merge and re-rank using a weighted combination of scores
3. Return final top-k results

This leverages the strengths of both methods!

In [ ]:
def hybrid_search(query: str,
                 embedding_model: SentenceTransformer,
                 corpus_embeddings: np.ndarray,
                 bm25_index: BM25,
                 chunks: List[Chunk],
                 top_k: int = 3,
                 dense_weight: float = 0.5) -> List[Tuple[Chunk, float]]:
    """Hybrid search combining dense and sparse retrieval.
    
    Args:
        dense_weight: Weight for dense scores (sparse weight = 1 - dense_weight)
    """
    # Get dense results
    dense_results = search(query, embedding_model, corpus_embeddings, chunks, top_k=top_k*2)
    
    # Get sparse results
    sparse_results = bm25_index.search(query, top_k=top_k*2)
    
    # Normalize scores to [0, 1]
    dense_scores = {i: score for i, (_, score) in enumerate(dense_results)}
    sparse_scores = {idx: score for idx, score in sparse_results}
    
    # Normalize dense scores
    if dense_scores:
        max_dense = max(dense_scores.values())
        dense_scores = {k: v/max_dense for k, v in dense_scores.items()}
    
    # Normalize sparse scores
    if sparse_scores:
        max_sparse = max(sparse_scores.values())
        if max_sparse > 0:
            sparse_scores = {k: v/max_sparse for k, v in sparse_scores.items()}
    
    # Combine scores
    combined_scores = {}
    all_indices = set(range(len(chunks)))
    
    for idx in all_indices:
        dense_score = dense_scores.get(idx, 0.0)
        sparse_score = sparse_scores.get(idx, 0.0)
        combined_scores[idx] = dense_weight * dense_score + (1 - dense_weight) * sparse_score
    
    # Get top-k
    top_indices = sorted(combined_scores.keys(), key=lambda x: combined_scores[x], reverse=True)[:top_k]
    
    return [(chunks[idx], combined_scores[idx]) for idx in top_indices]

print("Hybrid search function defined!")

### Compare All Three Retrieval Methods

Let's test dense, sparse, and hybrid retrieval side-by-side.

In [ ]:
test_query = "How do retrieval systems work in RAG?"

print(f"Query: '{test_query}'\n")
print("=" * 80)

# Dense
print("\n1. DENSE RETRIEVAL (Embeddings):")
dense_res = search(test_query, embedding_model, chunk_embeddings, all_chunks, top_k=3)
for i, (chunk, score) in enumerate(dense_res, 1):
    print(f"  {i}. [{score:.3f}] {chunk.doc_title}")

# Sparse
print("\n2. SPARSE RETRIEVAL (BM25):")
sparse_res = bm25.search(test_query, top_k=3)
for i, (idx, score) in enumerate(sparse_res, 1):
    print(f"  {i}. [{score:.2f}] {all_chunks[idx].doc_title}")

# Hybrid
print("\n3. HYBRID RETRIEVAL (Combined):")
hybrid_res = hybrid_search(test_query, embedding_model, chunk_embeddings, bm25, all_chunks, top_k=3)
for i, (chunk, score) in enumerate(hybrid_res, 1):
    print(f"  {i}. [{score:.3f}] {chunk.doc_title}")

print("\n" + "=" * 80)

## 10. Prompt Augmentation

### From Retrieval to Generation

Now we have relevant chunks - but how do we use them for generation? **Prompt augmentation**:

1. Retrieve top-k relevant chunks
2. Format them as context
3. Inject into the LLM prompt
4. Ask the model to answer based on this context

**Key principle**: Explicitly instruct the model to use the provided context and cite sources.

### Build RAG Prompt Template

Let's create a prompt template that structures retrieved context for optimal generation.

In [ ]:
def create_rag_prompt(query: str, retrieved_chunks: List[Tuple[Chunk, float]]) -> str:
    """Create a RAG prompt with retrieved context."""
    
    # Build context from retrieved chunks
    context_parts = []
    for i, (chunk, score) in enumerate(retrieved_chunks, 1):
        context_parts.append(f"[{i}] {chunk.doc_title}:\n{chunk.text}")
    
    context = "\n\n".join(context_parts)
    
    # Create structured prompt
    prompt = f"""You are a helpful AI assistant. Answer the question based on the provided context.

CONTEXT:
{context}

QUESTION:
{query}

INSTRUCTIONS:
- Answer based ONLY on the information in the context above
- If the context doesn't contain enough information, say so
- Cite sources using [1], [2], etc.
- Be concise and accurate

ANSWER:
"""
    
    return prompt

# Test prompt creation
test_query = "What is the transformer architecture?"
retrieved = search(test_query, embedding_model, chunk_embeddings, all_chunks, top_k=2)
prompt = create_rag_prompt(test_query, retrieved)

print("RAG Prompt Example:")
print("=" * 80)
print(prompt)
print("=" * 80)

## 11. Simple Generation Model

### Using GPT-2 for Generation

For demonstration, we'll use **GPT-2** (small) as our generation model. In production, you'd use larger models like GPT-4, Claude, or Llama.

Note: GPT-2 is quite limited compared to modern LLMs, but it demonstrates the RAG concept.

In [ ]:
# In production, you would load an LLM here
# For example:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# model = AutoModelForCausalLM.from_pretrained('gpt2')
# tokenizer = AutoTokenizer.from_pretrained('gpt2')

# For this demo, we'll simulate answers to avoid memory issues
print("Skipping LLM loading to keep notebook lightweight.")
print("In production, integrate with OpenAI, Anthropic, or local models.")

### Generation Function

Let's create a simple generation function that takes a prompt and produces text.

In [ ]:
# In production, you would load an LLM here
# For example:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# model = AutoModelForCausalLM.from_pretrained('gpt2')
# tokenizer = AutoTokenizer.from_pretrained('gpt2')

# For this demo, we'll simulate answers to avoid memory issues
print("Skipping LLM loading to keep notebook lightweight.")
print("In production, integrate with OpenAI, Anthropic, or local models.")

## 12. End-to-End RAG System

### Complete RAG Pipeline

Now let's put everything together into a complete RAG system!

In [ ]:
# In production, you would load an LLM here
# For example:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# model = AutoModelForCausalLM.from_pretrained('gpt2')
# tokenizer = AutoTokenizer.from_pretrained('gpt2')

# For this demo, we'll simulate answers to avoid memory issues
print("Skipping LLM loading to keep notebook lightweight.")
print("In production, integrate with OpenAI, Anthropic, or local models.")

### Test the RAG System

Let's ask our RAG system some questions and see how it performs!

In [ ]:
# Test question 1
result1 = rag_system.answer(
    "What is the transformer architecture?",
    top_k=2,
    retrieval_method='hybrid'
)
print("\n" + "="*80 + "\n")

In [ ]:
# Test question 2
result2 = rag_system.answer(
    "How does RAG help reduce hallucinations?",
    top_k=2,
    retrieval_method='hybrid'
)
print("\n" + "="*80 + "\n")

In [ ]:
# Test question 3
result3 = rag_system.answer(
    "What are vector databases?",
    top_k=2,
    retrieval_method='dense'
)
print("\n" + "="*80 + "\n")

## 13. Comparison: With vs Without RAG

### The RAG Advantage

Let's demonstrate RAG's value by comparing answers with and without retrieved context.

In [ ]:
# Demonstrate the RAG prompt structure
test_q = "What is attention mechanism in transformers?"

print("Question:", test_q)
print("\n" + "="*80)
print("\nWITHOUT RAG:")
print("-" * 80)
print("The LLM would only use its parametric memory (training data).")
print("This may lead to hallucinations or outdated information.")

print("\n" + "="*80)
print("\nWITH RAG (retrieval + generation):")
print("-" * 80)
rag_result = rag_system.answer(test_q, top_k=2, verbose=False)
print(f"Sources: {[c.doc_title for c, _ in rag_result['retrieved_chunks']]}")
print(f"\nPrompt length: {len(rag_result['prompt'])} characters")
print(f"\nSimulated Answer: {rag_result['answer']}")
print("\nThe LLM receives factual context, reducing hallucinations!")
print("\n" + "="*80)

## 14. Advanced: Re-ranking

### Improving Retrieval Quality

**Re-ranking** is a two-stage retrieval approach:

1. **First stage**: Fast retrieval (embedding similarity) to get top-k candidates (e.g., k=20)
2. **Second stage**: Expensive but accurate re-ranking of candidates (e.g., cross-encoder)

This balances speed and quality. We'll implement a simple re-ranker based on relevance scoring.

In [ ]:
def simple_rerank(query: str, 
                 candidates: List[Tuple[Chunk, float]],
                 top_k: int = 3) -> List[Tuple[Chunk, float]]:
    """Simple re-ranking based on term overlap."""
    
    query_terms = set(re.findall(r'\w+', query.lower()))
    
    reranked = []
    for chunk, original_score in candidates:
        # Count query term matches
        chunk_terms = set(re.findall(r'\w+', chunk.text.lower()))
        overlap = len(query_terms & chunk_terms)
        
        # Combine with original score
        new_score = original_score * 0.7 + (overlap / len(query_terms)) * 0.3
        reranked.append((chunk, new_score))
    
    # Sort by new score
    reranked.sort(key=lambda x: x[1], reverse=True)
    
    return reranked[:top_k]

# Test re-ranking
query = "transformer multi-head attention mechanism"
initial_results = search(query, embedding_model, chunk_embeddings, all_chunks, top_k=6)
reranked_results = simple_rerank(query, initial_results, top_k=3)

print(f"Query: '{query}'\n")
print("Initial retrieval (top 3):")
for i, (chunk, score) in enumerate(initial_results[:3], 1):
    print(f"  {i}. [{score:.3f}] {chunk.doc_title}")

print("\nAfter re-ranking:")
for i, (chunk, score) in enumerate(reranked_results, 1):
    print(f"  {i}. [{score:.3f}] {chunk.doc_title}")

## 15. Advanced: Hypothetical Document Embeddings (HyDE)

### Query Enhancement with HyDE

**HyDE** (Hypothetical Document Embeddings) is a clever trick:

1. Use LLM to generate a hypothetical answer to the query
2. Embed the hypothetical answer (not the query)
3. Search using this embedding

**Why it works**: Hypothetical answers are more similar to actual documents than short queries are.

Example:
- Query: "What is attention?"
- HyDE: "Attention is a mechanism that allows models to focus on relevant parts of input..."
- Better match to document embeddings!

In [ ]:
# In production, you would load an LLM here
# For example:
# from transformers import AutoModelForCausalLM, AutoTokenizer
# model = AutoModelForCausalLM.from_pretrained('gpt2')
# tokenizer = AutoTokenizer.from_pretrained('gpt2')

# For this demo, we'll simulate answers to avoid memory issues
print("Skipping LLM loading to keep notebook lightweight.")
print("In production, integrate with OpenAI, Anthropic, or local models.")

## 16. Evaluation Metrics

### Measuring RAG Quality

How do we know if our RAG system is good? Key metrics:

**Retrieval metrics**:
- **Recall@k**: What % of relevant docs are in top-k?
- **MRR** (Mean Reciprocal Rank): Average of 1/rank of first relevant result
- **NDCG** (Normalized Discounted Cumulative Gain): Relevance-weighted ranking quality

**Generation metrics**:
- **Faithfulness**: Does answer match retrieved context?
- **Relevance**: Does answer address the question?
- **Citation accuracy**: Are sources correctly attributed?

Let's implement simple evaluation:

In [ ]:
def evaluate_retrieval(query: str, 
                      relevant_doc_ids: List[str],
                      retrieved_chunks: List[Tuple[Chunk, float]]) -> Dict[str, float]:
    """Evaluate retrieval quality."""
    
    retrieved_doc_ids = [chunk.doc_id for chunk, _ in retrieved_chunks]
    
    # Recall@k: fraction of relevant docs retrieved
    relevant_retrieved = len(set(relevant_doc_ids) & set(retrieved_doc_ids))
    recall = relevant_retrieved / len(relevant_doc_ids) if relevant_doc_ids else 0.0
    
    # Precision@k: fraction of retrieved docs that are relevant
    precision = relevant_retrieved / len(retrieved_doc_ids) if retrieved_doc_ids else 0.0
    
    # MRR: 1/rank of first relevant document
    mrr = 0.0
    for i, doc_id in enumerate(retrieved_doc_ids, 1):
        if doc_id in relevant_doc_ids:
            mrr = 1.0 / i
            break
    
    return {
        'recall': recall,
        'precision': precision,
        'mrr': mrr
    }

# Test evaluation
test_cases = [
    {
        'query': 'What is transformer architecture?',
        'relevant_docs': ['doc1', 'doc2']  # Transformers and Attention
    },
    {
        'query': 'How do vector databases work?',
        'relevant_docs': ['doc5']  # Vector Databases
    },
    {
        'query': 'What are embeddings?',
        'relevant_docs': ['doc3']  # Embeddings
    }
]

print("Retrieval Evaluation:\n")
all_metrics = []
for case in test_cases:
    retrieved = search(case['query'], embedding_model, chunk_embeddings, all_chunks, top_k=3)
    metrics = evaluate_retrieval(case['query'], case['relevant_docs'], retrieved)
    all_metrics.append(metrics)
    
    print(f"Query: '{case['query']}'")
    print(f"  Recall@3: {metrics['recall']:.2f}")
    print(f"  Precision@3: {metrics['precision']:.2f}")
    print(f"  MRR: {metrics['mrr']:.2f}\n")

# Average metrics
avg_recall = np.mean([m['recall'] for m in all_metrics])
avg_precision = np.mean([m['precision'] for m in all_metrics])
avg_mrr = np.mean([m['mrr'] for m in all_metrics])

print(f"Average Performance:")
print(f"  Recall@3: {avg_recall:.2f}")
print(f"  Precision@3: {avg_precision:.2f}")
print(f"  MRR: {avg_mrr:.2f}")

## 17. Chunk Size Analysis

### Impact of Chunk Size on Retrieval

Let's experiment with different chunk sizes to understand the trade-off.

In [ ]:
# Test different chunk sizes
chunk_sizes = [100, 200, 300, 400]
results_by_size = {}

for size in chunk_sizes:
    # Re-chunk documents
    chunks = []
    for doc in documents:
        doc_chunks = chunk_text(doc['content'], doc['id'], doc['title'], 
                               max_chunk_size=size, overlap_sentences=1)
        chunks.extend(doc_chunks)
    
    # Re-embed
    chunk_texts = [c.text for c in chunks]
    embeddings = embedding_model.encode(chunk_texts, convert_to_numpy=True, show_progress_bar=False)
    
    # Test retrieval
    query = "What is the transformer architecture?"
    retrieved = search(query, embedding_model, embeddings, chunks, top_k=3)
    
    results_by_size[size] = {
        'num_chunks': len(chunks),
        'avg_chunk_len': np.mean([len(c.text) for c in chunks]),
        'top_score': retrieved[0][1] if retrieved else 0
    }

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Number of chunks
axes[0].bar(chunk_sizes, [results_by_size[s]['num_chunks'] for s in chunk_sizes])
axes[0].set_xlabel('Max Chunk Size')
axes[0].set_ylabel('Total Chunks')
axes[0].set_title('Number of Chunks vs Size')
axes[0].grid(True, alpha=0.3)

# Average chunk length
axes[1].bar(chunk_sizes, [results_by_size[s]['avg_chunk_len'] for s in chunk_sizes])
axes[1].set_xlabel('Max Chunk Size')
axes[1].set_ylabel('Avg Chunk Length')
axes[1].set_title('Average Chunk Length')
axes[1].grid(True, alpha=0.3)

# Retrieval quality
axes[2].bar(chunk_sizes, [results_by_size[s]['top_score'] for s in chunk_sizes])
axes[2].set_xlabel('Max Chunk Size')
axes[2].set_ylabel('Top-1 Similarity Score')
axes[2].set_title('Retrieval Quality (sample query)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nResults by chunk size:")
for size, metrics in results_by_size.items():
    print(f"  Size {size}: {metrics['num_chunks']} chunks, "
          f"avg {metrics['avg_chunk_len']:.0f} chars, "
          f"top score {metrics['top_score']:.3f}")

## 18. Production Considerations

### Scaling RAG Systems

Moving from prototype to production involves several considerations:

**1. Vector Database Selection**:
- **Pinecone**: Managed, easy to use, serverless
- **Weaviate**: Open-source, rich features, hybrid search
- **Milvus**: High performance, self-hosted
- **Chroma**: Simple, embedded, good for development
- **FAISS**: Library not service, extremely fast

**2. Indexing Strategies**:
- **HNSW** (Hierarchical Navigable Small World): Fast, accurate
- **IVF** (Inverted File Index): Memory efficient
- **Product Quantization**: Compress vectors

**3. Performance Optimization**:
- Batch embedding generation
- Caching frequent queries
- Async retrieval
- GPU acceleration

**4. Quality Improvements**:
- Better chunking (semantic splitting)
- Cross-encoder re-ranking
- Query expansion
- Metadata filtering

Let's visualize the production architecture:

In [ ]:
# Visualize production RAG architecture
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)

# Define components
components = [
    {'name': 'Documents', 'pos': (1, 8), 'color': 'lightblue'},
    {'name': 'Chunking\nPipeline', 'pos': (3, 8), 'color': 'lightgreen'},
    {'name': 'Embedding\nModel', 'pos': (5, 8), 'color': 'lightcoral'},
    {'name': 'Vector\nDatabase', 'pos': (7, 8), 'color': 'lightyellow'},
    {'name': 'User Query', 'pos': (1, 5), 'color': 'lightblue'},
    {'name': 'Query\nEmbedding', 'pos': (3, 5), 'color': 'lightcoral'},
    {'name': 'Retrieval\n+ Re-rank', 'pos': (5, 5), 'color': 'lightgreen'},
    {'name': 'LLM\nGeneration', 'pos': (7, 5), 'color': 'lightyellow'},
    {'name': 'Response', 'pos': (9, 5), 'color': 'lightblue'},
    {'name': 'Cache', 'pos': (5, 2), 'color': 'lavender'},
    {'name': 'Monitoring', 'pos': (7, 2), 'color': 'lavender'},
]

# Draw components
for comp in components:
    x, y = comp['pos']
    rect = plt.Rectangle((x-0.4, y-0.3), 0.8, 0.6, 
                         facecolor=comp['color'], edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x, y, comp['name'], ha='center', va='center', 
           fontsize=9, fontweight='bold')

# Draw arrows (indexing pipeline)
arrows_index = [
    ((1.4, 8), (2.6, 8)),
    ((3.4, 8), (4.6, 8)),
    ((5.4, 8), (6.6, 8)),
]

# Draw arrows (query pipeline)
arrows_query = [
    ((1.4, 5), (2.6, 5)),
    ((3.4, 5), (4.6, 5)),
    ((5.4, 5), (6.6, 5)),
    ((7.4, 5), (8.6, 5)),
]

# Draw connection from vector DB to retrieval
ax.annotate('', xy=(5, 5.3), xytext=(7, 7.7),
           arrowprops=dict(arrowstyle='->', lw=2, color='blue', linestyle='--'))

for arrow in arrows_index:
    ax.annotate('', xy=arrow[1], xytext=arrow[0],
               arrowprops=dict(arrowstyle='->', lw=2, color='black'))

for arrow in arrows_query:
    ax.annotate('', xy=arrow[1], xytext=arrow[0],
               arrowprops=dict(arrowstyle='->', lw=2, color='black'))

# Labels
ax.text(5, 9, 'INDEXING PIPELINE', ha='center', fontsize=12, fontweight='bold')
ax.text(5, 6, 'QUERY PIPELINE', ha='center', fontsize=12, fontweight='bold')

plt.title('Production RAG System Architecture', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

print("This is how a production RAG system is structured!")

## 19. FAISS Integration (Optional)

### Fast Approximate Nearest Neighbor Search

For large-scale systems (millions of vectors), we need efficient indexing. **FAISS** (Facebook AI Similarity Search) is the industry standard.

Let's integrate FAISS for faster retrieval:

In [ ]:
try:
    import faiss
    
    # Create FAISS index
    dimension = chunk_embeddings.shape[1]
    
    # Use flat L2 index (exact search)
    faiss_index = faiss.IndexFlatL2(dimension)
    
    # Add vectors to index
    faiss_index.add(chunk_embeddings.astype('float32'))
    
    print(f"FAISS index created with {faiss_index.ntotal} vectors")
    
    # Test search
    query = "What is attention mechanism?"
    query_embedding = embedding_model.encode(query, convert_to_numpy=True).astype('float32')
    
    # Search
    k = 3
    distances, indices = faiss_index.search(query_embedding.reshape(1, -1), k)
    
    print(f"\nQuery: '{query}'")
    print("\nFAISS Results:")
    for i, (idx, dist) in enumerate(zip(indices[0], distances[0]), 1):
        chunk = all_chunks[idx]
        # Convert L2 distance to similarity score (inverse)
        similarity = 1 / (1 + dist)
        print(f"{i}. [Similarity: {similarity:.3f}] {chunk.doc_title}")
        print(f"   {chunk.text[:100]}...\n")
    
    print("FAISS integration successful!")
    
except ImportError:
    print("FAISS not installed. Skipping FAISS demo.")
    print("To install: pip install faiss-cpu")

## 20. Key Takeaways

### What We've Learned

**Core RAG Concepts**:
1. **RAG solves three LLM problems**: knowledge cutoff, hallucinations, domain specificity
2. **Six-stage pipeline**: ingest → chunk → embed → index → retrieve → generate
3. **Chunking is critical**: balance between context and precision
4. **Embeddings capture meaning**: semantic similarity enables retrieval

**Retrieval Strategies**:
- **Dense retrieval**: Best for semantic understanding (embeddings)
- **Sparse retrieval**: Best for exact matches (BM25)
- **Hybrid retrieval**: Combines strengths of both
- **Re-ranking**: Two-stage approach for quality

**Advanced Techniques**:
- **HyDE**: Generate hypothetical answers for better retrieval
- **Query expansion**: Multiple queries for comprehensive retrieval
- **Metadata filtering**: Combine semantic and structured search

**Production Considerations**:
- **Vector databases**: Pinecone, Weaviate, Milvus, Chroma
- **Indexing algorithms**: HNSW, IVF, product quantization
- **Performance**: Caching, batching, async processing
- **Evaluation**: Recall, precision, MRR, faithfulness

**Key Insight**: RAG transforms LLMs from static knowledge bases into dynamic information systems that can access and reason over any corpus.

### Practical Guidelines

**When building a RAG system**:

1. **Start simple**: Dense retrieval with sentence-transformers
2. **Optimize chunks**: Experiment with sizes (150-300 chars is often good)
3. **Use hybrid retrieval**: Combine dense and sparse for robustness
4. **Evaluate continuously**: Track retrieval quality and generation faithfulness
5. **Consider re-ranking**: Especially for high-stakes applications
6. **Plan for scale**: Use vector databases from the start if you expect growth
7. **Prompt engineering matters**: Clear instructions reduce hallucinations
8. **Monitor in production**: Track query latency, retrieval quality, user feedback

**Common pitfalls to avoid**:
- Chunks too large (poor precision)
- Chunks too small (missing context)
- No overlap (broken context at boundaries)
- Wrong embedding model (use models trained for retrieval)
- Not normalizing scores when combining retrieval methods
- Forgetting to cite sources in prompts

### Summary Statistics

Let's review our RAG system's configuration and performance.

In [ ]:
# Summary statistics
print("RAG SYSTEM SUMMARY")
print("=" * 80)
print(f"\nKnowledge Base:")
print(f"  Documents: {len(documents)}")
print(f"  Total chunks: {len(all_chunks)}")
print(f"  Avg chunk size: {np.mean([len(c.text) for c in all_chunks]):.0f} characters")
print(f"  Embedding dimension: {chunk_embeddings.shape[1]}")
print(f"  Total embeddings: {chunk_embeddings.shape[0]}")
print(f"  Memory footprint: {chunk_embeddings.nbytes / 1024:.1f} KB")

print(f"\nModels:")
print(f"  Embedding model: all-MiniLM-L6-v2")
print(f"  Generation model: GPT-2")
print(f"  Device: {device}")

print(f"\nRetrieval Methods:")
print(f"  ✓ Dense (embedding similarity)")
print(f"  ✓ Sparse (BM25)")
print(f"  ✓ Hybrid (combined)")
print(f"  ✓ Re-ranking")
print(f"  ✓ HyDE")

print(f"\nCapabilities:")
print(f"  ✓ End-to-end question answering")
print(f"  ✓ Source attribution")
print(f"  ✓ Multiple retrieval strategies")
print(f"  ✓ Evaluation metrics")
print(f"  ✓ Production-ready architecture patterns")

print("\n" + "=" * 80)
print("\nRAG system complete! You now understand the full pipeline from")
print("document ingestion to answer generation. This is production-grade knowledge!")